In [1]:
import torch
import numpy as np

from utils import minmax_normalise_tensor, midpoint_to_box, upscale_box_tensor, mse

# Composite covariance function
from covariance import covariance_function, aggregate_base_covariance_matrix

# Load scene

Scene target tensor: [N, C, H, W]  

with channels: 
- `[:, 0, :, :]` bed
- `[:, 1, :, :]` surface
- `[:, 2, :, :]` thickness
- `[:, 3, :, :]` mask
- `[:, 4, :, :]` firn
- `[:, 5, :, :]` errorbed

and 
- `[:, 6, :, :]` y
- `[:, 7, :, :]` x

Define target tensor and auxiliary tensor.

## ToDos:
- Check correlation.
- Visualise both

In [2]:
scene_bed_tensor = torch.load('./torch_data/DOMEC_bed_scenes.pt')

## Check correlations of variables

Correlations are not clear: However, how are covariances related? 

In [3]:
# Flatten scenes and make variables columns
flat_vars = torch.cat((scene_bed_tensor[:, 0, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 1, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 2, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 4, :, :].reshape(-1).unsqueeze(0)), dim = 0)

torch.corrcoef(flat_vars)
# Bed and thickness are neg. corralated
# Surface and thickness are only 0.5
# Mask and bed are correlated: However, mask does not change too much over Dome C domain.

tensor([[ 1.0000, -0.2246, -0.9278, -0.4568],
        [-0.2246,  1.0000,  0.5719,  0.8969],
        [-0.9278,  0.5719,  1.0000,  0.7279],
        [-0.4568,  0.8969,  0.7279,  1.0000]])

In [4]:
ground_truth = minmax_normalise_tensor(scene_bed_tensor[0, 0, :, :]).unsqueeze(0)
ground_truth.shape

torch.cat((scene_bed_tensor[0, 6, :, :].unsqueeze(0), 
           scene_bed_tensor[0, 7, :, :].unsqueeze(0)), dim = 0)

# upscale_box_tensor()

tensor([[[-557000., -557000., -557000.,  ..., -557000., -557000., -557000.],
         [-557500., -557500., -557500.,  ..., -557500., -557500., -557500.],
         [-558000., -558000., -558000.,  ..., -558000., -558000., -558000.],
         ...,
         [-578000., -578000., -578000.,  ..., -578000., -578000., -578000.],
         [-578500., -578500., -578500.,  ..., -578500., -578500., -578500.],
         [-579000., -579000., -579000.,  ..., -579000., -579000., -579000.]],

        [[ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.],
         [ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.],
         [ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.],
         ...,
         [ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.],
         [ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.],
         [ 899000.,  899500.,  900000.,  ...,  920000.,  920500.,  921000.]]])

In [15]:
up_factor = 5
# For loop first and wrap up into a function once it works
loss_list = []
baseline_loss_list = []

# for each scene
for i in range(0, 1):
    # Normalise each target scene and create explicit first dim, e.g. torch.Size([1, 45, 45])
    target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)

    # Generate box channels for target scene
    target_box_channels = midpoint_to_box(torch.cat((scene_bed_tensor[i, 6, :, :].unsqueeze(0), 
                                                     scene_bed_tensor[i, 7, :, :].unsqueeze(0)), dim = 0))
    
    # ground truth box: Concatenate bed_elevation Channel and mid_points
    target_ground_truth_box = torch.cat((target_ground_truth, target_box_channels), dim = 0)
    
    # Upscale to generate low-res. input
    # Low-res. 
    lr_bed = upscale_box_tensor(target_ground_truth_box, upscaling_factor = up_factor)
    # Subset last 4 channels to get the box channels of the low-resolution grid
    lr_box_channels = lr_bed[-4:, :, :]

    # Normalisation
    # High-resolution auxiliary data to compute the base_covariance
    hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

    # aligned-grid lines scenario: target_hr_grid already defined above

    ### Base covariance ###
    # In non-aligning grids this is the covariance matrix on the AUXILIARY grid not on the target grid
    # Covariance function: No need to use true spatial coordinates as we can add these channels afterwards. Parameters like the lengthscale operate on the normalised space
    base_covariance = covariance_function(hr_aux)
    # Flatten from 2D to 1D, cast new dim and repeat
    row_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-1).repeat(1, 1, base_covariance.shape[-1])
    # .unsqueeze(-2) creates explicit dimension we wanna copy across (middle dimension)
    column_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-2).repeat(1, base_covariance.shape[-1], 1)

    # generate base covariance box with Channel 0: covar, Channel 1:4: row, Channel 5:8: column box channels. 
    base_covar_box = torch.cat((torch.tensor(base_covariance).unsqueeze(0), row_box_channels, column_box_channels), dim = 0)

    k_ah_al_tensor, k_al_al_tensor = aggregate_base_covariance_matrix(base_covar_box, lr_box_channels)

    ### MEAN RECONSTRUCTION ###
    hr_inferred = predictive_mean(lr_bed[0, :, :].unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)

    ### LOSS ###
    mse_scene = mse(hr_inferred, target_ground_truth.squeeze())
    # append loss to list above. append is in place
    loss_list.append(mse(hr_inferred, target_ground_truth.squeeze()).numpy().item())

    ### BASELINE ###

    ### BASELINE LOSS ###
    baseline_loss_list.append(mse(hr_inferred, target_ground_truth.squeeze()).numpy().item())



In [16]:
loss_list

[0.0001081543762300085]

In [6]:
def predictive_mean(lr_variable_channel, k_ah_al, k_al_al, noise = 0.05, mu = 0.5):
    """ Take in argumnets to compute predictive mean and output inferred mean channel

    Args:
        lr_variable_channel (_type_): _description_
        k_ah_al (_type_): _description_
        k_al_al (_type_): _description_
        noise (float, optional): _description_. Defaults to 0.05.
        mu (float, optional): _description_. Defaults to 0.5.
    """
    # Mean channel: zero(ish) mean target_lowres grid e.g. torch.Size([1, 9, 9])
    pl_al_minus_mu = lr_variable_channel - torch.ones(size = lr_variable_channel.shape) * mu  

    # Weights
    W = torch.matmul(k_ah_al, torch.linalg.inv(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise))) # torch.Size([2025, 81])

    # Get shape from W
    # Error with data type
    hr_inference_adjusted = torch.div(torch.matmul(W, pl_al_minus_mu.reshape(-1).unsqueeze(1).double()), torch.matmul(W, torch.ones(size = (W.shape[-1], 1)).double())) + mu

    # Cast into 2D shape
    hr_inference_2D = hr_inference_adjusted.reshape(int(np.sqrt(hr_inference_adjusted.shape[0])), -1)

    return(hr_inference_2D)

## Questions:

- Normalisation outside or inside
- Mean function

Limitation:
- Transferability of covariance likely depends on scale: new proof of concept for 